In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# 1. Load Data and Artifacts

In [2]:
df = pd.read_csv("../dataset//Crop_recommendation.csv")
df = df.drop(columns=[c for c in df.columns if c.lower().startswith('unnamed')])
df.columns = [c.lower() for c in df.columns]

In [3]:
X = df.drop('label', axis=1)
y = df['label']

In [4]:
scaler = joblib.load("../models/scaler.pkl")
selected_features = joblib.load("../models/selected_features.pkl")

In [5]:
print(X.head())

    n   p   k  temperature   humidity        ph    rainfall
0  90  42  43    20.879744  82.002744  6.502985  202.935536
1  85  58  41    21.770462  80.319644  7.038096  226.655537
2  60  55  44    23.004459  82.320763  7.840207  263.964248
3  74  35  40    26.491096  80.158363  6.980401  242.864034
4  78  42  42    20.130175  81.604873  7.628473  262.717340


In [6]:
print(scaler.feature_names_in_)

['n' 'p' 'k' 'temperature' 'humidity' 'ph' 'rainfall']


In [7]:
print(selected_features)

['n', 'p', 'k', 'temperature', 'humidity', 'ph', 'rainfall']


In [10]:
# 1. Clean your dataframe columns just in case (to match the names inside the scaler)
X.columns = X.columns.str.lower().str.strip()

# 2. Force X to have the EXACT same columns and order the scaler expects
X = X[scaler.feature_names_in_] #it shuffles the columns

# 3. Now the transform will work perfectly
X_scaled = scaler.transform(X)

# 2. Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. Define Model Zoo

In [12]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(kernel='rbf')
}

In [13]:
print(models.items())

dict_items([('LogisticRegression', LogisticRegression(max_iter=2000)), ('KNN', KNeighborsClassifier()), ('DecisionTree', DecisionTreeClassifier(random_state=42)), ('RandomForest', RandomForestClassifier(n_estimators=200, random_state=42)), ('SVM', SVC())])


# 4. Training Loop

In [14]:
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"Trained: {name}")

Trained: LogisticRegression
Trained: KNN
Trained: DecisionTree
Trained: RandomForest
Trained: SVM


In [15]:
print(trained_models)

{'LogisticRegression': LogisticRegression(max_iter=2000), 'KNN': KNeighborsClassifier(), 'DecisionTree': DecisionTreeClassifier(random_state=42), 'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42), 'SVM': SVC()}


# 5. Save Models

In [16]:
for name, model in trained_models.items():
    joblib.dump(model, f"../models/{name}.pkl")